In [52]:
import pandas as pd
import numpy as np
import glob
import os

PATHS = {
    "AT": {
        "raw":       "../data/AT/raw_data/intraday_epex/trades/",
        "processed": "../data/AT/processed/",
    },
    "DE": {
        "raw":       "../data/DE/raw_data/intraday_epex/trades/",
        "processed": "../data/DE/processed/",
    },
}

for market in PATHS.values():
    os.makedirs(market["processed"], exist_ok=True)

In [53]:
def load_raw_trades(raw_path: str, market_label: str) -> pd.DataFrame:
    
    files = glob.glob(os.path.join(raw_path, "**/*.csv"), recursive=True)
    
    if not files:
        raise FileNotFoundError(f"No CSV files found under {raw_path}")
    
    print(f"[{market_label}] Found {len(files)} files — loading...")
    
    frames = []
    for f in files:
        try:
            # dtype=str loads EVERYTHING as string first
            # This prevents pandas from silently mis-casting
            # columns like price="-12.58" or self_trade="false"
            frames.append(pd.read_csv(f, dtype=str))
        except Exception as e:
            print(f"  Skipped {f}: {e}")
    
    df = pd.concat(frames, ignore_index=True)
    df["market"] = market_label
    
    print(f"[{market_label}] Raw shape: {df.shape}")
    return df


df_at = load_raw_trades(PATHS["AT"]["raw"], "AT")
df_de = load_raw_trades(PATHS["DE"]["raw"], "DE")

[AT] Found 9762 files — loading...
[AT] Raw shape: (3664611, 14)
[DE] Found 7705 files — loading...
[DE] Raw shape: (9056883, 14)


In [56]:
DOMESTIC_AREAS = {
    "AT": {"10YAT-APG------L"},
    "DE": {
        "10YDE-VE-------2",    # TenneT DE
        "10YDE-RWENET---I",    # Amprion
        "10YDE-EON------1",    # E.ON (now TenneT)
        "10Y1001A1001A63L",    # 50Hertz
    },
}


def clean_trades(df: pd.DataFrame, market_label: str) -> pd.DataFrame:
    
    df = df.copy()

    # ── 2a. Timestamps ──────────────────────────────────────────────────────
    # Raw format: "2023-12-31T19:07:08.429+01:00"
    # utc=True normalises the +01:00 offset to UTC internally,
    # then we convert to Europe/Vienna so CET/CEST is handled correctly.
    df["time"] = (
    pd.to_datetime(df["time"], format="ISO8601", utc=True)
      .dt.tz_convert("Europe/Vienna")
    )

    # ── 2b. Numeric columns ─────────────────────────────────────────────────
    # errors="coerce" turns any unparseable string (e.g. empty, "N/A") → NaN
    # instead of crashing. We catch those in 2d below.
    df["price"]    = pd.to_numeric(df["price"],    errors="coerce")
    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

    # ── 2c. Boolean — your CSV has lowercase "true"/"false" ─────────────────
    # .str.strip() removes any accidental whitespace
    # == "true" produces a proper bool Series
    df["self_trade"] = df["self_trade"].str.strip().str.lower() == "true"

    # ── 2d. Verify and drop constant columns ────────────────────────────────
    # "revision" and "state" appear constant in your data.
    # We verify first — if they're NOT constant, we keep them.
    for col, expected in [("state", "EXECUTED"), ("revision", "1")]:
        unique_vals = df[col].unique()
        if len(unique_vals) == 1 and str(unique_vals[0]) == expected:
            df = df.drop(columns=[col])
            print(f"[{market_label}] Dropped constant column: '{col}' (always '{expected}')")
        else:
            print(f"[{market_label}] Kept '{col}' — has values: {unique_vals}")

    # "currency" and "quantity_unit" and "exchange" are also likely constant
    for col in ["currency", "quantity_unit", "exchange"]:
        unique_vals = df[col].unique()
        if len(unique_vals) == 1:
            df = df.drop(columns=[col])
            print(f"[{market_label}] Dropped constant column: '{col}' (always '{unique_vals[0]}')")
        else:
            print(f"[{market_label}] Kept '{col}' — has values: {unique_vals}")

    # ── 2e. Drop rows we cannot use ─────────────────────────────────────────
    before = len(df)

    df = df.dropna(subset=["price", "quantity", "time"])  # missing core fields
    df = df[df["quantity"] > 0]                           # zero-MW trades are noise
    df = df.drop_duplicates(subset=["trade_id"])          # overlapping CSV exports

    dropped = before - len(df)
    print(f"[{market_label}] Dropped {dropped:,} bad/duplicate rows ({dropped/before:.1%})")

    # ── 2f. Engineer useful columns ─────────────────────────────────────────
    df["date"]        = df["time"].dt.date
    df["hour"]        = df["time"].dt.hour
    df["minute"]      = df["time"].dt.minute
    df["day_of_week"] = df["time"].dt.dayofweek   # 0=Monday, 6=Sunday
    df["week"]        = df["time"].dt.isocalendar().week.astype(int)
    df["month"]       = df["time"].dt.month
    df["year"]        = df["time"].dt.year
    df["is_weekend"]  = df["day_of_week"] >= 5

    # Cross-border: buyer and seller are in different price zones
    df["cross_border"] = df["buy_delivery_area"] != df["sell_delivery_area"]

    # Domestic: at least one side of the trade is in the home market
    domestic = DOMESTIC_AREAS[market_label]
    df["is_domestic"] = (
        df["buy_delivery_area"].isin(domestic) |
        df["sell_delivery_area"].isin(domestic)
    )

    # Traded value in EUR — useful for VWAP calculations later
    df["traded_value_eur"] = df["price"] * df["quantity"]

    print(f"[{market_label}] Final shape: {df.shape}\n")
    return df


df_at = clean_trades(df_at, "AT")
df_de = clean_trades(df_de, "DE")

[AT] Dropped constant column: 'state' (always 'EXECUTED')
[AT] Dropped constant column: 'revision' (always '1')
[AT] Dropped constant column: 'currency' (always 'EUR')
[AT] Dropped constant column: 'quantity_unit' (always 'MW')
[AT] Dropped constant column: 'exchange' (always 'EPEX')
[AT] Dropped 0 bad/duplicate rows (0.0%)
[AT] Final shape: (3664611, 20)

[DE] Dropped constant column: 'state' (always 'EXECUTED')
[DE] Dropped constant column: 'revision' (always '1')
[DE] Dropped constant column: 'currency' (always 'EUR')
[DE] Dropped constant column: 'quantity_unit' (always 'MW')
[DE] Dropped constant column: 'exchange' (always 'EPEX')
[DE] Dropped 0 bad/duplicate rows (0.0%)
[DE] Final shape: (9056883, 20)



In [57]:
def sanity_check(df: pd.DataFrame, market_label: str):
    print(f"\n{'='*55}")
    print(f"  SANITY CHECK — {market_label}")
    print(f"{'='*55}")
    print(f"  Rows              : {len(df):,}")
    print(f"  Columns           : {list(df.columns)}")
    print(f"  Date range        : {df['time'].min().date()}  →  {df['time'].max().date()}")
    print(f"  Unique contracts  : {df['contract_id'].nunique():,}")
    print(f"  Price range       : {df['price'].min():.2f}  →  {df['price'].max():.2f} €/MWh")
    print(f"  Negative prices   : {(df['price'] < 0).mean():.2%}")
    print(f"  Cross-border      : {df['cross_border'].mean():.2%}")
    print(f"  Self trades       : {df['self_trade'].mean():.2%}")
    print(f"  Domestic trades   : {df['is_domestic'].mean():.2%}")
    
    # Check for any remaining nulls
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"  Nulls             : none ✓")
    else:
        print(f"  Nulls             :\n{nulls}")
    print()


sanity_check(df_at, "AT")
sanity_check(df_de, "DE")


  SANITY CHECK — AT
  Rows              : 3,664,611
  Columns           : ['contract_id', 'time', 'trade_id', 'buy_delivery_area', 'sell_delivery_area', 'price', 'quantity', 'self_trade', 'market', 'date', 'hour', 'minute', 'day_of_week', 'week', 'month', 'year', 'is_weekend', 'cross_border', 'is_domestic', 'traded_value_eur']
  Date range        : 2023-12-31  →  2024-07-05
  Unique contracts  : 15,678
  Price range       : -1919.00  →  2000.01 €/MWh
  Negative prices   : 7.03%
  Cross-border      : 65.31%
  Self trades       : 0.44%
  Domestic trades   : 100.00%
  Nulls             : none ✓


  SANITY CHECK — DE
  Rows              : 9,056,883
  Columns           : ['contract_id', 'time', 'trade_id', 'buy_delivery_area', 'sell_delivery_area', 'price', 'quantity', 'self_trade', 'market', 'date', 'hour', 'minute', 'day_of_week', 'week', 'month', 'year', 'is_weekend', 'cross_border', 'is_domestic', 'traded_value_eur']
  Date range        : 2023-12-31  →  2024-04-08
  Unique contracts  :

In [58]:
def save_processed(df: pd.DataFrame, market_label: str):
    out_path = os.path.join(
        PATHS[market_label]["processed"],
        f"intraday_trades_{market_label}.parquet"
    )
    df.to_parquet(out_path, index=False, engine="pyarrow", compression="snappy")
    
    size_mb = os.path.getsize(out_path) / 1e6
    print(f"[{market_label}] Saved → {out_path}  ({size_mb:.1f} MB)")


save_processed(df_at, "AT")
save_processed(df_de, "DE")

[AT] Saved → ../data/AT/processed/intraday_trades_AT.parquet  (69.8 MB)
[DE] Saved → ../data/DE/processed/intraday_trades_DE.parquet  (168.5 MB)
